# Day 61: Introduction to Deep Learning
## From Perceptron to Deep Neural Networks

---

# PART 1: THEORY (50 minutes)

## 1. What is Deep Learning?

Deep Learning is a subset of Machine Learning that uses **neural networks with many layers** to learn patterns from data.

```
AI -> ML -> DL
```

**The key difference from traditional ML:** Deep Learning automatically learns features from raw data. You don't need to manually engineer features.

| | Traditional ML | Deep Learning |
|---|---|---|
| Feature Engineering | Manual (human picks features) | Automatic (network learns them) |
| Data Requirement | Works with small data | Needs LOTS of data |
| Performance | Good on tabular data | Best on images, text, audio |
| Interpretability | Easy to understand | Black box |
| Training Speed | Fast (seconds-minutes) | Slow (minutes-days) |
| Hardware | CPU | GPU preferred |

## 2. The Biological Inspiration

Your brain has ~86 billion neurons connected by synapses. Each neuron:
1. Receives electrical signals from other neurons (inputs)
2. If the total signal is strong enough, it **fires** (threshold)
3. Sends its signal to other neurons (output)

**Artificial Neuron (Perceptron):**
```
output = activation(w1*x1 + w2*x2 + ... + wn*xn + b)
```
- **Inputs (x):** Data features
- **Weights (w):** How important is each input?
- **Bias (b):** Threshold for firing
- **Activation function:** Decides the output value

## 3. The History of Deep Learning

| Year | Milestone |
|------|-----------|
| 1958 | Perceptron (Frank Rosenblatt) — first artificial neuron |
| 1969 | Minsky proves perceptrons can't solve XOR — first "AI winter" |
| 1986 | Backpropagation popularized (Rumelhart, Hinton) — revival |
| 2006 | Deep Belief Networks (Hinton) — start of modern DL era |
| 2012 | AlexNet wins ImageNet by huge margin — CNN revolution |
| 2014 | GANs invented (Goodfellow) — generative AI begins |
| 2017 | Transformers paper (Vaswani et al.) — "Attention is All You Need" |
| 2022 | ChatGPT, Stable Diffusion — DL goes mainstream |
| 2024+ | GPT-4, Claude, multimodal AI — DL everywhere |

## 4. When to Use Deep Learning

**USE Deep Learning:**
- Images (classification, detection, segmentation)
- Natural Language (translation, chatbots, summarization)
- Audio/Speech (recognition, synthesis)
- Video (action recognition, tracking)
- Complex patterns with large datasets

**DON'T Use Deep Learning:**
- Tabular data with < 10,000 rows (use XGBoost/Random Forest)
- When you need interpretability (use Decision Trees, Logistic Regression)
- When you need fast training (use simpler models)
- When you don't have a GPU (even CPU training is slow)

## 5. Frameworks — TensorFlow/Keras vs PyTorch

| | TensorFlow/Keras | PyTorch |
|---|---|---|
| Made by | Google | Meta |
| Style | Declarative (define then run) | Imperative (run as you define) |
| Learning Curve | Easier (Keras API) | Steeper |
| Production | TF Serving, TF Lite | TorchServe, ONNX |
| Research | Less common now | Dominant |

**We'll use Keras** — it's the most beginner-friendly way to learn DL concepts.

---

# PART 2: PRACTICAL (50 minutes)

## 6. Setting Up Your Environment

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
plt.rcParams['figure.figsize'] = (10, 6)

try:
    import tensorflow as tf
    from tensorflow import keras
    from tensorflow.keras import layers
    print(f"TensorFlow version: {tf.__version__}")
    print(f"Keras version: {keras.__version__}")
    gpus = tf.config.list_physical_devices('GPU')
    if gpus:
        print(f"GPU Available: {len(gpus)} device(s)")
        for gpu in gpus:
            print(f"  - {gpu}")
    else:
        print("No GPU found — running on CPU (slower but works fine for learning)")
except ImportError:
    print("TensorFlow not installed!")
    print("Run: pip install tensorflow")
    import sys; sys.exit(1)


## 7. Your First Neural Network — The Perceptron

Let's build a single neuron to solve a simple problem: classify whether a number is > 5 based on two features.

In [ ]:
# Generate simple data
np.random.seed(42)
X = np.random.randn(200, 2) * 2
y = (X[:, 0] + X[:, 1] > 0).astype(int)  # 1 if sum > 0, else 0

# Visualize
plt.figure(figsize=(10, 5))
plt.scatter(X[y==0, 0], X[y==0, 1], c='red', label='Class 0', alpha=0.6, s=50)
plt.scatter(X[y==1, 0], X[y==1, 1], c='blue', label='Class 1', alpha=0.6, s=50)
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.title('Data: Class 1 = f1 + f2 > 0')
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
# Build a single neuron (perceptron) model
model = keras.Sequential([
    layers.Dense(1, activation='sigmoid', input_shape=(2,))
])

model.compile(optimizer=keras.optimizers.SGD(learning_rate=0.1),
              loss='binary_crossentropy',
              metrics=['accuracy'])

model.summary()

print(f"\nThis model has only {model.count_params():,} trainable parameters!")
print(f"Weight (2) + Bias (1) = 3 parameters total")


In [ ]:
# Train the perceptron
history = model.fit(X, y, epochs=50, validation_split=0.2, verbose=0)

# Plot learning curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(history.history['loss'], label='Train Loss', linewidth=2)
axes[0].plot(history.history['val_loss'], label='Val Loss', linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Loss Over Training')
axes[0].legend()
axes[0].grid(True)

axes[1].plot(history.history['accuracy'], label='Train Accuracy', linewidth=2)
axes[1].plot(history.history['val_accuracy'], label='Val Accuracy', linewidth=2)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Accuracy Over Training')
axes[1].legend()
axes[1].grid(True)
plt.tight_layout()
plt.show()

loss, acc = model.evaluate(X, y, verbose=0)
print(f"Final Accuracy: {acc:.3f} ({acc*100:.1f}%)")


## 8. Perceptron Limitation — XOR Problem

A single perceptron can ONLY solve linearly separable problems. Let's prove it.

In [ ]:
# XOR data — NOT linearly separable!
X_xor = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
y_xor = np.array([0, 1, 1, 0])

# Train perceptron on XOR
model_xor = keras.Sequential([
    layers.Dense(1, activation='sigmoid', input_shape=(2,))
])
model_xor.compile(optimizer='sgd', loss='binary_crossentropy', metrics=['accuracy'])
history_xor = model_xor.fit(X_xor, y_xor, epochs=200, verbose=0)

loss, acc = model_xor.evaluate(X_xor, y_xor, verbose=0)
print(f"XOR Accuracy with perceptron: {acc:.1%}")

if acc < 0.8:
    print("FAILED! A single neuron CANNOT solve XOR.")
    print("This is why we need MULTIPLE layers — deep networks!")
else:
    print("Worked!")


In [ ]:
# XOR with hidden layer (now it works!)
model_xor2 = keras.Sequential([
    layers.Dense(4, activation='relu', input_shape=(2,)),  # Hidden layer!
    layers.Dense(1, activation='sigmoid')
])
model_xor2.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
history_xor2 = model_xor2.fit(X_xor, y_xor, epochs=200, verbose=0)

loss, acc = model_xor2.evaluate(X_xor, y_xor, verbose=0)
print(f"XOR Accuracy with hidden layer: {acc:.1%}")
print("The hidden layer transforms the data so it becomes linearly separable!")


---

# PART 3: EXERCISES (20 minutes)

In [ ]:
# Exercise 1: Try different numbers of neurons in the hidden layer
# What happens with 2 neurons? 8 neurons? 16 neurons?
for n_neurons in [2, 4, 8, 16]:
    m = keras.Sequential([
        layers.Dense(n_neurons, activation='relu', input_shape=(2,)),
        layers.Dense(1, activation='sigmoid')
    ])
    m.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    m.fit(X_xor, y_xor, epochs=100, verbose=0)
    _, acc = m.evaluate(X_xor, y_xor, verbose=0)
    print(f"Hidden neurons: {n_neurons:3d} -> Accuracy: {acc:.1%}")


In [ ]:
# Exercise 2: Build a network for the Iris dataset
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

iris = load_iris()
X_iris, y_iris = iris.data, iris.target
X_tr, X_te, y_tr, y_te = train_test_split(X_iris, y_iris, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_tr_s = scaler.fit_transform(X_tr)
X_te_s = scaler.transform(X_te)

# TODO: Build a model with 2 hidden layers to classify iris
# Hint: Input=4, Output=3 (softmax), loss='sparse_categorical_crossentropy'
print("Iris data loaded. Build your model!")
print(f"Train: {X_tr_s.shape}, Test: {X_te_s.shape}")


## Key Takeaways

- DL = neural networks with **many layers**
- A **neuron** computes: activation(w*x + b)
- **Keras** makes building networks easy with `Sequential` API
- Single perceptron can't solve XOR — need **hidden layers**
- DL excels at **images, text, audio**; traditional ML for tabular data
- **GPU** dramatically speeds up DL training
- TensorFlow/Keras is the most beginner-friendly framework

**Tomorrow:** Neural Network fundamentals — layers, activation functions, and building deeper networks!